In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
sys.path.append("/Users/samuel/Código/TFG")

# Preprocessing
We must process the data to make it suitable to work with

## Read the inventory generated in the previous phase

In [4]:
import pandas as pd
import librosa as lib
import soundfile as sf
import src.io
import src.inventory
import src.preprocess
import scipy.signal as sig
import re
import math
import numpy as np

In [5]:
inv = pd.read_csv("../data/interim/inventory.csv")
inv

,file,dataset,patient,position,hash,sample_rate,channels,bit_depth,is_float,peak,rms,dc_offset,clipping,duration,lung_label,heart_label,disease,strat_key,group,fold
0,L0014.wav,HLS-CMDS,HLS-Mix13,LUA,6e16fbcfbcae5accbba8bcd344b617e3e20ade4749f727...,4000,1,16,False,0.034668,0.005713,-1.188151e-06,0.000000,15.000,Rhonchi,Late Diastolic Murmur,NaN,HLS-L:Abnormal|H:Abnormal,HLS-HS14,0
1,L0028.wav,HLS-CMDS,HLS-Mix27,RC,ae5100abccff274c0aa3b159fbbec03741a3405774b404...,4000,1,16,False,0.046295,0.005200,3.433736e-06,0.000000,15.000,Coarse Crackles,AV Block,NaN,HLS-L:Abnormal|H:Abnormal,HLS-HS14,0
2,M0005.wav,HLS-CMDS,HLS-Mix4,RUSB,ab5b5fa88f88471147005fffb0d8a400040098d9eb55f1...,4000,1,16,False,0.091583,0.006620,7.553101e-07,0.000000,15.000,Fine Crackles,AV Block,NaN,HLS-L:Abnormal|H:Abnormal,HLS-HS12,2
3,H0140.wav,HLS-CMDS,HLS-Mix139,RLA,8dcaf616ccb1aa74674c7a29f6d16e5ee91c0d5e3b2af1...,4000,1,16,False,0.198608,0.007852,1.919047e-06,0.000000,15.000,Normal,Early Systolic Murmur,NaN,HLS-L:Normal|H:Abnormal,HLS-LS16,4
4,M0011.wav,HLS-CMDS,HLS-Mix10,Apex,f67c2e00075f912635272294c9ad195a282876b8f0ef35...,4000,1,16,False,0.019775,0.003763,3.550212e-07,0.000000,15.000,Normal,S4,NaN,HLS-L:Normal|H:Abnormal,HLS-HS13,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,"DP79_asthma,E W,A R U,46,M.wav",NIH,NIH79,A R U,cf22c72a9ec1c7a69a9a9003eb0689b5637fe58d8b347a...,4000,1,16,False,0.999969,0.074263,-4.895800e-03,0.000012,20.336,NaN,NaN,asthma,NIH:asthma,NIH79,0
867,"DP69_pneumonia,Bronchial,P R L ,64,M.wav",NIH,NIH69,P R L,f52208b3dc08e6ca908c425c57d1272312c864fac15195...,4000,1,16,False,1.000000,0.064400,1.221145e-03,0.000010,24.176,NaN,NaN,pneumonia,NIH:pneumonia,NIH69,4
868,"BP13_Heart Failure,C,P L L,55,M.wav",NIH,NIH13,P L L,3cdd5ad5094809551f58b129207c4b5e31fae066d2e2a6...,4000,1,16,False,1.000000,0.174402,-1.121447e-04,0.000051,14.576,NaN,NaN,heart failure,NIH:heart failure,NIH13,3
869,"DP107_Asthma,E W,P L U,59,F.wav",NIH,NIH107,P L U,90f55d4ff96dc2fed44ae5f947a0066f42ef6c8630f37a...,4000,1,16,False,0.999969,0.105253,2.235783e-03,0.000040,12.656,NaN,NaN,asthma,NIH:asthma,NIH107,2


## Resampling
4kHz has been chosen as the resampling frequency because it is the sample_rate of all the files in the current datasets. If later more datasets were to be added, they would be resampled to this frequency

In [ ]:
conf = src.io.open_config()
target_sr = conf["sample-rate"]
for n, i in inv.iterrows():
    dataset, gr = src.inventory.search_row(i)
    path = f"{conf[dataset][gr]}/{i.file}"
    file, sample_rate = lib.load(path, sr=None)
    type = src.inventory.Type.match_file(dataset, gr)
    # print(f"Resampling {i.file}")
    resampled_file = src.preprocess.resample(file, sample_rate, target_sr, conf)
    # print(f"Normalizing {i.file}")
    # print(f"pre: {i.rms}")
    normalized_file = src.preprocess.normalize(resampled_file)



../data/raw/HLS-CMDS/Mix/L0014.wav
../data/raw/HLS-CMDS/Mix/L0028.wav
../data/raw/HLS-CMDS/Mix/M0005.wav
../data/raw/HLS-CMDS/Mix/H0140.wav
../data/raw/HLS-CMDS/Mix/M0011.wav
../data/raw/HLS-CMDS/Mix/M0039.wav
../data/raw/HLS-CMDS/Mix/H0020.wav
../data/raw/HLS-CMDS/Mix/H0034.wav
../data/raw/HLS-CMDS/Mix/H0008.wav
../data/raw/HLS-CMDS/Mix/H0009.wav
../data/raw/HLS-CMDS/Mix/H0035.wav
../data/raw/HLS-CMDS/Mix/H0021.wav
../data/raw/HLS-CMDS/Mix/M0038.wav
../data/raw/HLS-CMDS/Mix/H0141.wav
../data/raw/HLS-CMDS/Mix/M0010.wav
../data/raw/HLS-CMDS/Mix/M0004.wav
../data/raw/HLS-CMDS/Mix/L0029.wav
../data/raw/HLS-CMDS/Mix/L0001.wav
../data/raw/HLS-CMDS/Mix/L0015.wav
../data/raw/HLS-CMDS/Mix/L0003.wav
../data/raw/HLS-CMDS/Mix/L0017.wav
../data/raw/HLS-CMDS/Mix/M0012.wav
../data/raw/HLS-CMDS/Mix/H0143.wav
../data/raw/HLS-CMDS/Mix/M0006.wav
../data/raw/HLS-CMDS/Mix/H0037.wav
../data/raw/HLS-CMDS/Mix/H0023.wav
../data/raw/HLS-CMDS/Mix/H0022.wav
../data/raw/HLS-CMDS/Mix/H0036.wav
../data/raw/HLS-CMDS